<a href="https://colab.research.google.com/github/seekff/learn-python/blob/main/%E5%8D%8F%E7%A8%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**为什么需要协程？**

多线程/多进程在高并发下会遇到 C10K 问题（同时一万个请求连接），进程大量上下文切换、资源消耗大。

事件循环（event loop）模型更轻量，NGINX 就是典型代表。

协程 + async/await 提供了 可读性强、性能高 的并发方式。

**Python 协程的基础概念**

① async 定义异步函数

    async def func():
      ...

调用不会立即执行，而是返回 coroutine object。

② await 用于挂起等待

await 会“阻塞当前协程”，但不会阻塞整个程序。

await 后面必须是可等待对象（如另一个协程、asyncio.sleep）。

③ asyncio.run(main())

Python 3.7+ 的推荐入口

自动创建/关闭事件循环

整个程序生命周期只调用一次

In [16]:
import time


#同步
def crawl_page(url):
  print('crawling {}'.format(url))
  sleep_time = int(url.split('_')[-1])
  time.sleep(sleep_time)
  print('OK {}'.format(url))

def main(urls):
  for url in urls:
    crawl_page(url)

%time main(['url_1','url_2','url_3','url_4'])

print('----上面是同步，下面是协程------')



crawling url_1
OK url_1
crawling url_2
OK url_2
crawling url_3
OK url_3
crawling url_4
OK url_4
CPU times: user 4.75 ms, sys: 357 µs, total: 5.11 ms
Wall time: 10 s
----上面是同步，下面是协程------


**协程 vs 同步：爬虫示例**

同步版本爬 4 个页面：10 秒  

协程版本（但使用 await 顺序执行）：仍然 10 秒

→ 因为 await 是同步等待。

    import asyncio

    #协程
    async def crawl_page(url):
      print('crawling {}'.format(url))
      sleep_time = int(url.split('_')[-1])
      await asyncio.sleep(sleep_time)
      print('OK {}'.format(url))

    async def main(urls):
      for url in urls:
        await crawl_page(url)

    %time asyncio.run(main(['url_1','url_2','url_3','url_4']))

    crawling url_1
    OK url_1
    crawling url_2
    OK url_2
    crawling url_3
    OK url_3
    crawling url_4
    OK url_4
    CPU times: user 10 s, sys: 0 ns, total: 10 s
    Wall time: 10 s

**真正的并发：Task（任务）**

使用 asyncio.create_task() 才能让协程并发执行：


    tasks = [asyncio.create_task(crawl_page(url)) for url in urls]
    for task in tasks:
      await task
结果：总耗时 ≈ 最慢的那个任务（约 4 秒）

→ 真正实现并发。

也可用更简洁的：

    await asyncio.gather(*tasks)
例子：

    import asyncio


    async def crawl_page(url):
      print('crawling {}'.format(url))
      sleep_time = int(url.split('_')[-1])
      await asyncio.sleep(sleep_time)
      print('OK {}'.format(url))

    async def main(urls):
      tasks = [asyncio.create_task(crawl_page(url)) for url in urls]
      for task in tasks:
        await task
      #await asyncio.gather(*tasks)  #更简洁写法


    %time asyncio.run(main(['url_1','url_2','url_3','url_4']))


    crawling url_1
    crawling url_2
    crawling url_3
    crawling url_4
    OK url_1
    OK url_2
    OK url_3
    OK url_4
    CPU times: user 4.01 s, sys: 0 ns, total: 4.01 s
    Wall time: 4.01 s


In [20]:
import asyncio
import time # 导入 time 模块用于手动计时


async def crawl_page(url):
  print('crawling {}'.format(url))
  sleep_time = int(url.split('_')[-1])
  await asyncio.sleep(sleep_time)
  print('OK {}'.format(url))

async def main(urls):
  tasks = [asyncio.create_task(crawl_page(url)) for url in urls]
  # for task in tasks:
  #   await task
  await asyncio.gather(*tasks)


# 手动计时，避免与 %%time 结合导致 'SyntaxError'，同时避免 'asyncio.run()' 导致的 'RuntimeError'
start_time = time.time()
await main(['url_1','url_2','url_3','url_4'])
end_time = time.time()
print(f"Wall time: {end_time - start_time:.3f} seconds")

crawling url_1
crawling url_2
crawling url_3
crawling url_4
OK url_1
OK url_2
OK url_3
OK url_4
Wall time: 4.002 seconds


In [ ]:
import asyncio


async def worker_1():
  print('worker_1 start')
  await asyncio.sleep(1)
  print('worker_1 done')


async def worker_2():
  print('worker_2 start')
  await asyncio.sleep(2)
  print('worker_2 done')

async def main():
  print('befor await')
  await worker_1()
  await worker_2()
  print('after await')

%time asyncio.run(main())

befor await
worker_1 start
worker_1 done
worker_2 start
worker_2 done
after await
CPU times: user 3.02 s, sys: 0 ns, total: 3.02 s
Wall time: 3.02 s

In [ ]:
import asyncio

async def worker_1():
  print('worker_1 start')
  await asyncio.sleep(1)
  print('worker_1 done')

async def worker_2():
  print('worker_2 start')
  await asyncio.sleep(2)
  print('worker_2 done')

async def main():
  task1 = asyncio.create_task(worker_1())
  task2 = asyncio.create_task(worker_2())
  print('before await')
  await task1
  print('awaited worker_1')
  await task2
  print('awaited worker_2')

%time asyncio.run(main())

before await
worker_1 start
worker_2 start
worker_1 done
awaited worker_1
worker_2 done
awaited worker_2
CPU times: user 2.01 s, sys: 0 ns, total: 2.01 s
Wall time: 2.01 s


1、asyncio.run(main())，程序进入 main() 函数，事件循环开启；

2、task1 和 task2 任务被创建，并进入事件循环等待运行；运行到 print，输出 'before await'；

3、await task1 执行，用户选择从当前的主任务中切出，事件调度器开始调度 worker_1；

4、worker_1 开始运行，运行 print 输出'worker_1 start'，然后运行到 await asyncio.sleep(1)， 从当前任务切出，事件调度器开始调度 worker_2；

5、worker_2 开始运行，运行 print 输出 'worker_2 start'，然后运行 await asyncio.sleep(2) 从当前任务切出；

6、以上所有事件的运行时间，都应该在 1ms 到 10ms 之间，甚至可能更短，事件调度器从这个时候开始暂停调度；

7、一秒钟后，worker_1 的 sleep 完成，事件调度器将控制权重新传给 task_1，输出 'worker_1 done'，task_1 完成任务，从事件循环中退出；

8、await task1 完成，事件调度器将控制器传给主任务，输出 'awaited worker_1'，·然后在 await task2 处继续等待；

9、两秒钟后，worker_2 的 sleep 完成，事件调度器将控制权重新传给 task_2，输出 'worker_2 done'，task_2 完成任务，从事件循环中退出；

10、主任务输出 'awaited worker_2'，协程全任务结束，事件循环结束。






In [24]:
import asyncio

async def worker_1():
  await asyncio.sleep(1)
  return 1

async def worker_2():
  await asyncio.sleep(2)
  return 2/0

async def worker_3():
  await asyncio.sleep(3)
  return 3

async def main():
  task1 = asyncio.create_task(worker_1())
  task2 = asyncio.create_task(worker_2())
  task3 = asyncio.create_task(worker_3())

  await asyncio.sleep(2)
  task3.cancel()

  res = await asyncio.gather(task1,task2,task3,return_exceptions=True)
  print(res)

%time asyncio.run(main())

[1, ZeroDivisionError('division by zero'), CancelledError('')]
Wall time: 2.001 seconds


**协程运行机制（事件循环）**

事件循环负责：

调度任务

在 await 处切出

在 I/O 完成后切回

保证单线程内的“伪并发”

页面中通过 worker_1 / worker_2 的例子详细展示了事件循环的调度过程。

**超时与异常处理**

使用 asyncio.gather(..., return_exceptions=True)：

不会因为某个任务异常而取消其他任务

返回列表中包含正常结果、异常对象、取消对象

任务取消：

    task.cancel()

**协程版生产者-消费者模型**

使用 asyncio.Queue() 实现：

producer 使用 await queue.put()

consumer 使用 await queue.get()

完整展示了协程在并发模型中的优势

In [27]:
import asyncio
import random

async def comsumer(queue, id):
  while True:
    val = await queue.get()
    print('{} get a val: {}'.format(id, val))
    await asyncio.sleep(1)

async def producer(queue, id):
  for i in range(5):
    val = random.randint(1,10)
    await queue.put(val)
    print('{} put a val: {}'.format(id, val))
    await asyncio.sleep(1)

async def main():
  queue = asyncio.Queue()

  consumer_1 = asyncio.create_task(comsumer(queue, 'consumer_1'))
  consumer_2 = asyncio.create_task(comsumer(queue, 'consumer_2'))
  producer_1 = asyncio.create_task(producer(queue, 'producer_1'))
  producer_2 = asyncio.create_task(producer(queue, 'producer_2'))

  await asyncio.sleep(10)

  consumer_1.cancel()
  consumer_2.cancel()

  await asyncio.gather(consumer_1, consumer_2, producer_1, producer_2, return_exceptions=True)

%time asyncio.run(main())

producer_1 put a val: 9
producer_2 put a val: 2
consumer_1 get a val: 9
consumer_2 get a val: 2
producer_1 put a val: 9
producer_2 put a val: 1
consumer_1 get a val: 9
consumer_2 get a val: 1
producer_1 put a val: 8
producer_2 put a val: 3
consumer_1 get a val: 8
consumer_2 get a val: 3
producer_1 put a val: 8
producer_2 put a val: 7
consumer_1 get a val: 8
consumer_2 get a val: 7
producer_1 put a val: 3
producer_2 put a val: 2
consumer_1 get a val: 3
consumer_2 get a val: 2
Wall time: 10.006 seconds


**豆瓣“即将上映”电影爬虫**

对比同步 vs 协程版本：

同步：56 秒

协程：约 5 秒

协程版本使用：

aiohttp 异步请求

gather 并发抓取多个电影详情页

BeautifulSoup 解析页面

显著提升性能。

In [35]:
import requests
from bs4 import BeautifulSoup

def main():
  url = "https://movie.douban.com/cinema/later/beijing/"
  init_page = requests.get(url).content
  init_soup = BeautifulSoup(init_page, 'lxml')

  all_movies = init_soup.find('div', id="showing-soon")
  for each_movie in all_movies.find_all('div', class_="item"):
      all_a_tag = each_movie.find_all('a')
      all_li_tag = each_movie.find_all('li')

      movie_name = all_a_tag[1].text
      url_to_fetch = all_a_tag[1]['href']
      movie_date = all_li_tag[0].text

      response_item = requests.get(url_to_fetch).content
      soup_item = BeautifulSoup(response_item, 'lxml')
      img_tag = soup_item.find('img')

      print('{} {} {}'.format(movie_name, movie_date, img_tag['src']))

%time main()

b''


AttributeError: 'NoneType' object has no attribute 'find_all'

In [37]:
import asyncio
import aiohttp

from bs4 import BeautifulSoup

# Define headers for aiohttp request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

async def fetch_content(url):
    async with aiohttp.ClientSession(
        headers=headers, connector=aiohttp.TCPConnector(ssl=False)
    ) as session:
        async with session.get(url) as response:
            response.raise_for_status() # Raise an exception for bad status codes
            return await response.text()

async def main():
    url = "https://movie.douban.com/cinema/later/beijing/"
    init_page = await fetch_content(url)
    init_soup = BeautifulSoup(init_page, 'lxml')

    movie_names, urls_to_fetch, movie_dates = [], [], []

    all_movies = init_soup.find('div', id="showing-soon")

    if all_movies is None:
        print("Error: Could not find the 'showing-soon' section on the page. The page structure might have changed.")
        return

    for each_movie in all_movies.find_all('div', class_="item"):
        all_a_tag = each_movie.find_all('a')
        all_li_tag = each_movie.find_all('li')

        if len(all_a_tag) > 1 and len(all_li_tag) > 0:
            movie_names.append(all_a_tag[1].text)
            urls_to_fetch.append(all_a_tag[1]['href'])
            movie_dates.append(all_li_tag[0].text)
        else:
            print(f"Warning: Skipping a movie item due to insufficient 'a' or 'li' tags in {each_movie}")

    tasks = [fetch_content(url) for url in urls_to_fetch]
    pages = await asyncio.gather(*tasks, return_exceptions=True) # Collect exceptions

    for movie_name, movie_date, page in zip(movie_names, movie_dates, pages):
        if isinstance(page, Exception):
            print(f"Error fetching detail for {movie_name}: {page}")
            continue

        soup_item = BeautifulSoup(page, 'lxml')
        img_tag = soup_item.find('img')

        if img_tag and 'src' in img_tag.attrs:
            print('{} {} {}'.format(movie_name, movie_date, img_tag['src']))
        else:
            print(f'Warning: Could not find image src for {movie_name} or img tag is missing on detail page.')

# Use await main() directly in Colab to run on the existing event loop
import time
start_time = time.time()
await main()
end_time = time.time()
print(f"Wall time: {end_time - start_time:.3f} seconds")

Wall time: 10.846 seconds


**协程是 单线程并发，由用户定何时切换（await）。**

async/await + create_task 是 Python 现代异步编程的核心。

协程适合 I/O 密集型任务（爬虫、网络请求）。

写协程时要理解事件循环的调度逻辑。

工程中选择协程/线程/进程需根据场景权衡

**协程核心机制**

 **1. 定义协程函数**
    async def func():
      await something()

特点：

- async 定义异步函数

- 调用不会执行 → 返回 coroutine object

**2. 创建协程对象**

    coro = func()

此时仍未执行，只是“任务蓝图”

**3. 事件循环启动**

    asyncio.run(main())

作用：

- 创建事件循环

- 调度所有任务

- 程序生命周期内只调用一次

**4. await 的作用 **

    await coro

含义：

- 当前协程让出控制权

- 事件循环切换去执行其他任务

- I/O 完成后再切回来继续执行

**5. Task（任务） **

    task = asyncio.create_task(coro)

作用：

- 让协程真正“并发”执行

- 不会阻塞当前协程

- 事件循环会自动调度它

**6. gather 并发收集 **

    await asyncio.gather(*tasks)

作用：

- 并发执行多个任务

- 等所有任务完成后返回结果

**7. 协程调度流程**

事件循环不断重复：

1. 选择一个可运行的 Task
2. 执行到第一个 await
3. await 触发 I/O → 挂起当前 Task
4. 切换到下一个 Task
5. I/O 完成后恢复 Task
6. 所有 Task 完成 → 事件循环结束

**8. 超时 & 异常处理**

    await asyncio.gather(*tasks, return_exceptions=True)

- 不会因为一个任务异常导致全部取消

- 返回列表中包含正常值、异常对象、取消对象

**9. 协程的本质总结 **

✔ 单线程并发

✔ 用户决定何时切换（await）

✔ 事件循环负责调度  

✔ 适合 I/O 密集型任务

✔ 写法比线程更简洁、可控  

一句话记住协程

**协程 = 单线程 + 可控切换点（await）+ 事件循环调度 + Task 实现并发。**